In [ ]:
import os
import sys

os.environ["USE_TORCHVISION"] = "0"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

if getattr(sys.modules.get("torchvision"), "__spec__", None) is None:
    sys.modules.pop("torchvision", None)

!apt-get update -y && !apt-get install -y ffmpeg
!pip uninstall -y -q torchvision torchao
!pip install -q transformers==4.46.3 accelerate==1.1.0 datasets soundfile librosa jiwer evaluate scikit-learn torchaudio peft inflect

print("\n--> Packages & Environment Ready for 5% LoRA + Dual-Encoder/Decoder Skip Model Training!")

In [ ]:
import torch
import gc

if 'model' in globals():
    del model
if 'processor' in globals():
    del processor

gc.collect()
if hasattr(torch, "cuda") and hasattr(torch.cuda, "empty_cache"):
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> System Processing Unit: {device}")
if device.type == "cuda":
    print(f"--> GPU Device: {torch.cuda.get_device_name(0)}")
    try:
        print(f"--> Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    except Exception:
        print("--> Initial VRAM Allocated: N/A")

In [ ]:
import io
import re
import string
import datasets
import numpy as np
import librosa
import inflect
from datasets import load_dataset

datasets.config.TORCHCODEC_AVAILABLE = False
p = inflect.engine()

print("--> Loading ATCOSIM and approved public ATC datasets from Hugging Face...")

try:
    atcosim_dataset = load_dataset("Jzuluaga/atcosim_corpus")
    atcosim_dataset = atcosim_dataset.cast_column("audio", datasets.Audio(decode=False))
    atcosim_train = atcosim_dataset["train"]
    atcosim_test = atcosim_dataset["test"]
except Exception as e:
    print(f"Error loading ATCOSIM: {e}")
    atcosim_train, atcosim_test = None, None

try:
    uwb_dataset = load_dataset("Jzuluaga/uwb_atcc", split="train")
    uwb_dataset = uwb_dataset.cast_column("audio", datasets.Audio(decode=False))
except Exception as e:
    print(f"Error loading UWB-ATCC: {e}")
    uwb_dataset = None

try:
    atco2_dataset = load_dataset("Jzuluaga/atco2_corpus_1h", split="train")
except Exception:
    try:
        atco2_dataset = load_dataset("Jzuluaga/atco2_corpus_1h", split="test")
    except Exception as e:
        print(f"Error loading ATCO2-1h: {e}")
        atco2_dataset = None

if atco2_dataset is not None:
    atco2_dataset = atco2_dataset.cast_column("audio", datasets.Audio(decode=False))

print(f"\n--> ATCOSIM Train: {len(atcosim_train) if atcosim_train else 0} | Test: {len(atcosim_test) if atcosim_test else 0}")
print(f"--> UWB-ATCC: {len(uwb_dataset) if uwb_dataset else 0}")
print(f"--> ATCO2-1h: {len(atco2_dataset) if atco2_dataset else 0}")

def clean_atc_transcript(text):
    if text is None:
        return ""
    text = text.lower()
    text = ''.join(char for char in text if char not in string.punctuation)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def apply_spectral_subtraction(noisy_audio, sr=16000, alpha=0.3):
    n_fft = 2048
    hop_length = 512
    noisy_spectrogram = librosa.stft(noisy_audio, n_fft=n_fft, hop_length=hop_length)
    noisy_mag = np.abs(noisy_spectrogram)
    noisy_phase = np.angle(noisy_spectrogram)
    noise_frames = max(1, int(0.1 * sr / hop_length))
    noise_power_spectrum = np.mean(noisy_mag[:, :noise_frames]**2, axis=1, keepdims=True)
    clean_mag = np.maximum(noisy_mag**2 - alpha * noise_power_spectrum, 0)
    clean_mag = np.sqrt(clean_mag)
    denoised_audio = librosa.istft(clean_mag * np.exp(1j * noisy_phase))
    return denoised_audio.astype(np.float32)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class QuadDualSkipNoveltyWhisperHooks(nn.Module):
    def __init__(self, model):
        super().__init__()
        base = model.base_model.model if hasattr(model, "base_model") else model
        encoder = base.model.encoder if hasattr(base, "model") else base.encoder
        decoder = base.model.decoder if hasattr(base, "model") else base.decoder

        enc_device = encoder.layers[16].weight.device if hasattr(encoder.layers[16], 'weight') else encoder.conv1.weight.device

        self.enc_conv1_out = None
        self.enc_maxpool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.gamma_conv_novelty = nn.Parameter(torch.tensor(0.1, device=enc_device))

        def enc_conv1_hook(module, input, output):
            self.enc_conv1_out = F.gelu(output)
            return output

        def enc_conv2_hook(module, input, output):
            z = output
            if self.enc_conv1_out is not None:
                y = self.enc_maxpool(self.enc_conv1_out)
                if y.size(2) != z.size(2):
                    y = F.interpolate(y, size=z.size(2), mode='linear', align_corners=False)
                return F.gelu(z + self.gamma_conv_novelty.to(z.device) * y)
            return F.gelu(z)

        encoder.conv1.register_forward_hook(enc_conv1_hook)
        encoder.conv2.register_forward_hook(enc_conv2_hook)

        self.enc_early_idx_1 = 0
        self.enc_late_idx_1 = 16
        self.enc_early_idx_2 = 8
        self.enc_late_idx_2 = 24

        self.gamma_enc1 = nn.Parameter(torch.tensor(0.05, device=enc_device))
        self.gamma_enc2 = nn.Parameter(torch.tensor(0.05, device=enc_device))

        self.enc_out_1 = None
        self.enc_out_2 = None

        def enc_hook_early_1(module, input, output):
            self.enc_out_1 = output[0] if isinstance(output, tuple) else output
            return output

        def enc_hook_late_1(module, input, output):
            z = output[0] if isinstance(output, tuple) else output
            y = self.enc_out_1
            if y is not None:
                if y.device != z.device:
                    y = y.to(z.device)
                if y.size(1) != z.size(1):
                    y_trans = y.permute(0, 2, 1)
                    z_trans = z.permute(0, 2, 1)
                    y_aligned = F.interpolate(y_trans, size=z_trans.size(2), mode='linear', align_corners=False).permute(0, 2, 1)
                else:
                    y_aligned = y
                z = z + self.gamma_enc1.to(z.device) * y_aligned
            if isinstance(output, tuple):
                return (z,) + output[1:]
            return z

        def enc_hook_early_2(module, input, output):
            self.enc_out_2 = output[0] if isinstance(output, tuple) else output
            return output

        def enc_hook_late_2(module, input, output):
            z = output[0] if isinstance(output, tuple) else output
            y = self.enc_out_2
            if y is not None:
                if y.device != z.device:
                    y = y.to(z.device)
                if y.size(1) != z.size(1):
                    y_trans = y.permute(0, 2, 1)
                    z_trans = z.permute(0, 2, 1)
                    y_aligned = F.interpolate(y_trans, size=z_trans.size(2), mode='linear', align_corners=False).permute(0, 2, 1)
                else:
                    y_aligned = y
                z = z + self.gamma_enc2.to(z.device) * y_aligned
            if isinstance(output, tuple):
                return (z,) + output[1:]
            return z

        encoder.layers[self.enc_early_idx_1].register_forward_hook(enc_hook_early_1)
        encoder.layers[self.enc_late_idx_1].register_forward_hook(enc_hook_late_1)
        encoder.layers[self.enc_early_idx_2].register_forward_hook(enc_hook_early_2)
        encoder.layers[self.enc_late_idx_2].register_forward_hook(enc_hook_late_2)

        self.dec_early_idx_1 = 0
        self.dec_late_idx_1 = 16
        self.dec_early_idx_2 = 8
        self.dec_late_idx_2 = 24

        dec_device = decoder.layers[16].weight.device if hasattr(decoder.layers[16], 'weight') else enc_device

        self.gamma_dec1 = nn.Parameter(torch.tensor(0.05, device=dec_device))
        self.gamma_dec2 = nn.Parameter(torch.tensor(0.05, device=dec_device))

        self.dec_out_1 = None
        self.dec_out_2 = None

        def dec_hook_early_1(module, input, output):
            self.dec_out_1 = output[0] if isinstance(output, tuple) else output
            return output

        def dec_hook_late_1(module, input, output):
            z = output[0] if isinstance(output, tuple) else output
            y = self.dec_out_1
            if y is not None:
                if y.device != z.device:
                    y = y.to(z.device)
                if y.size(1) != z.size(1):
                    y_trans = y.permute(0, 2, 1)
                    z_trans = z.permute(0, 2, 1)
                    y_aligned = F.interpolate(y_trans, size=z_trans.size(2), mode='linear', align_corners=False).permute(0, 2, 1)
                else:
                    y_aligned = y
                z = z + self.gamma_dec1.to(z.device) * y_aligned
            if isinstance(output, tuple):
                return (z,) + output[1:]
            return z

        def dec_hook_early_2(module, input, output):
            self.dec_out_2 = output[0] if isinstance(output, tuple) else output
            return output

        def dec_hook_late_2(module, input, output):
            z = output[0] if isinstance(output, tuple) else output
            y = self.dec_out_2
            if y is not None:
                if y.device != z.device:
                    y = y.to(z.device)
                if y.size(1) != z.size(1):
                    y_trans = y.permute(0, 2, 1)
                    z_trans = z.permute(0, 2, 1)
                    y_aligned = F.interpolate(y_trans, size=z_trans.size(2), mode='linear', align_corners=False).permute(0, 2, 1)
                else:
                    y_aligned = y
                z = z + self.gamma_dec2.to(z.device) * y_aligned
            if isinstance(output, tuple):
                return (z,) + output[1:]
            return z

        decoder.layers[self.dec_early_idx_1].register_forward_hook(dec_hook_early_1)
        decoder.layers[self.dec_late_idx_1].register_forward_hook(dec_hook_late_1)
        decoder.layers[self.dec_early_idx_2].register_forward_hook(dec_hook_early_2)
        decoder.layers[self.dec_late_idx_2].register_forward_hook(dec_hook_late_2)

print("--> [SUCCESS] Complete Audio Conv Novelty + Quad Encoder/Decoder Skip Hooks Class Defined!")

In [ ]:
import os
import sys

os.environ["USE_TORCHVISION"] = "0"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

if getattr(sys.modules.get("torchvision"), "__spec__", None) is None:
    sys.modules.pop("torchvision", None)

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

model_id = 'openai/whisper-large-v2'

print(f"--> Loading Pretrained Processor & Model: {model_id}...")
processor = WhisperProcessor.from_pretrained(model_id)

model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

target_modules = [
    "q_proj", "v_proj", "k_proj", "out_proj",
    "fc1", "fc2", "self_attn.q_proj", "self_attn.v_proj",
    "encoder_attn.q_proj", "encoder_attn.v_proj"
]

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(model, lora_config)
model.skip_hook = QuadDualSkipNoveltyWhisperHooks(model)

for param in model.skip_hook.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_percentage = (trainable_params / total_params) * 100

print(f"\n--> [SUCCESS] 5% LoRA & Quad Skip Connections Integrated Successfully!")
print(f"--> Total Parameters        : {total_params:,}")
print(f"--> Trainable Parameters    : {trainable_params:,}")
print(f"--> Trainable Weight Ratio  : {trainable_percentage:.2f}%")

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=1e-5)

def generic_collate_fn(batch):
    features_list = []
    text_list = []
    for sample in batch:
        if sample is None:
            continue
        audio_field = sample.get("audio")
        text_field = sample.get("text") or sample.get("sentence") or sample.get("transcript")
        if audio_field is None or text_field is None:
            continue
        try:
            if "bytes" in audio_field:
                audio_bytes = audio_field["bytes"]
                raw_audio, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)
            elif "path" in audio_field:
                raw_audio, sr = librosa.load(audio_field["path"], sr=16000)
            else:
                continue

            denoised_audio = apply_spectral_subtraction(raw_audio, sr=16000, alpha=0.3)

            inputs = processor.feature_extractor(
                denoised_audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding="max_length",
                max_length=3000,
                truncation=True
            )
            feat = inputs.input_features.squeeze(0)
            if feat.shape[-1] != 3000:
                continue
            features_list.append(feat)
            text_list.append(text_field)
        except Exception:
            continue

    if len(features_list) == 0:
        return None, None

    input_features_batch = torch.stack(features_list)
    labels = processor.tokenizer(
        text_list,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return input_features_batch, labels

def train_dataset_epochs(dataset_obj, dataset_name, num_epochs=3):
    if dataset_obj is None:
        print(f"--> Dataset {dataset_name} is not available. Skipping.", flush=True)
        return

    model.train()
    loader = DataLoader(dataset_obj, batch_size=8, shuffle=True, collate_fn=generic_collate_fn, drop_last=False)

    for epoch in range(num_epochs):
        print(f"\n==================================================", flush=True)
        print(f"--> STARTING EPOCH {epoch + 1}/{num_epochs} FOR DATASET: {dataset_name} (Batches: {len(loader)})", flush=True)
        print(f"==================================================", flush=True)

        step = 0
        for input_features, labels in loader:
            if input_features is None or labels is None:
                continue
            try:
                input_features = input_features.to(model.device, dtype=torch.float16)
                labels = labels.to(model.device)
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                step += 1
                if step % 50 == 0:
                    print(f"[{dataset_name} | Epoch {epoch+1}] Step {step} | Loss: {loss.item():.4f}", flush=True)
            except Exception:
                continue

    print(f"--> {dataset_name} ki 3-epoch training successfully complete ho gayi hai!", flush=True)

train_dataset_epochs(atcosim_train, "ATCOSIM Train Set", num_epochs=3)
train_dataset_epochs(uwb_dataset, "UWB-ATCC Corpus", num_epochs=3)
train_dataset_epochs(atco2_dataset, "ATCO2 Test 1h Subset", num_epochs=3)

print("\n--> ATCOSIM TRAIN + UWB + ATCO2 3-EPOCH TRAINING COMPLETE!")

In [ ]:
import sys
import torch
import gc
import io
import re
import librosa
import warnings
from collections import Counter
from jiwer import wer as jiwer_wer
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
model.eval()

EVAL_BATCH_SIZE = 5

comprehensive_icao_prompt = (
    "speedbird seven seven heavy climb and maintain flight level two eight zero "
    "descend and maintain altitude one zero thousand squawk one five zero two "
    "radar contact turn left heading one eight zero cleared ils runway zero nine right "
    "wind zero eight zero at one two gusts two zero cleared to land line up and wait "
    "hold short runway two seven report established on localizer contact tower on one one eight point one "
    "descend now to altitude four thousand feet qh one zero one three set course direct to karlsruhe "
    "report passing flight level two five zero squawk ident continue present heading until advised"
)
prompt_ids = torch.tensor(processor.get_prompt_ids(comprehensive_icao_prompt)).to(device)

digit_map = {
    '0': 'zero', '1': 'one', '2': 'two', '3': 'three', '4': 'four',
    '5': 'five', '6': 'six', '7': 'seven', '8': 'eight', '9': 'nine'
}

num_words = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
num_pattern = '|'.join(num_words)

def convert_digits_to_words(text):
    if not text:
        return ""
    text = re.sub(r'(\d)', r' \1 ', text)
    def replace_single_digit(match):
        d = match.group(0)
        return digit_map.get(d, d)
    text = re.sub(r'\d', replace_single_digit, text)
    for _ in range(3):
        text = re.sub(f'({num_pattern})({num_pattern})', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def eval_collate_fn(batch):
    raw_audios = []
    texts = []
    for sample in batch:
        try:
            audio_bytes = sample["audio"]["bytes"]
            raw_audio, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)
            denoised_audio = apply_spectral_subtraction(raw_audio, sr=16000, alpha=0.3)
            raw_audios.append(denoised_audio)
            texts.append(sample["text"])
        except Exception:
            continue
    return raw_audios, texts

def calculate_metrics(predicted_text, actual_text):
    norm_pred = clean_atc_transcript(predicted_text)
    norm_actual = clean_atc_transcript(actual_text)

    predicted_words = norm_pred.split()
    actual_words = norm_actual.split()

    if len(actual_words) == 0:
        acc = 1.0 if len(predicted_words) == 0 else 0.0
        prec = 1.0 if len(predicted_words) == 0 else 0.0
        rec = acc
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        wer_val = 0.0 if len(predicted_words) == 0 else 1.0
        ser_val = int(norm_pred != norm_actual)
        return acc, prec, rec, f1, wer_val, ser_val

    pred_counts = Counter(predicted_words)
    actual_counts = Counter(actual_words)

    true_positives = sum(min(pred_counts[word], actual_counts[word]) for word in pred_counts)
    false_positives = max(0, len(predicted_words) - true_positives)
    false_negatives = max(0, len(actual_words) - true_positives)

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
    recall = min(1.0, true_positives / len(actual_words))
    accuracy = recall
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    wer = float(jiwer_wer(norm_actual, norm_pred)) if norm_pred else 1.0
    ser = int(norm_pred != norm_actual)
    return accuracy, precision, recall, f1_score, wer, ser

subsets_to_evaluate = [100, 200, 500]

for limit in subsets_to_evaluate:
    print(f"\n==================================================", flush=True)
    print(f"EVALUATING 5% LoRA + QUAD-SKIP MODEL ON UNSEEN ATCOSIM TEST SUBSET: {limit} SAMPLES (NUM_BEAMS=5)", flush=True)
    print(f"==================================================", flush=True)

    subset_data = atcosim_test.select(range(min(limit, len(atcosim_test))))
    test_loader = DataLoader(
        subset_data,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=eval_collate_fn,
        num_workers=0
    )

    predicted_transcripts = []
    ground_truth_test_list = []
    processed_count = 0

    gc.collect()
    if hasattr(torch, "cuda") and hasattr(torch.cuda, "empty_cache"):
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    with torch.no_grad():
        for raw_audios, texts in test_loader:
            if not raw_audios:
                continue
            inputs = processor(raw_audios, sampling_rate=16000, return_tensors="pt", padding=True)
            input_features = inputs.input_features.to(device, dtype=torch.float16)
            attention_mask = inputs.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)

            predicted_ids = model.generate(
                input_features=input_features,
                attention_mask=attention_mask,
                prompt_ids=prompt_ids,
                num_beams=5,
                language="english",
                task="transcribe",
                use_cache=True
            )

            raw_transcriptions = processor.batch_decode(predicted_ids, skip_special_tokens=True)

            for raw_txt, gt_txt in zip(raw_transcriptions, texts):
                converted_txt = convert_digits_to_words(raw_txt)
                clean_pred = clean_atc_transcript(converted_txt)
                clean_gt = clean_atc_transcript(gt_txt)

                predicted_transcripts.append(clean_pred)
                ground_truth_test_list.append(clean_gt)
                processed_count += 1
                print(f"[{processed_count}/{limit}] PRED: {clean_pred} | GT: {clean_gt}", flush=True)
                sys.stdout.flush()

            del inputs, input_features, predicted_ids
            if hasattr(torch, "cuda") and hasattr(torch.cuda, "empty_cache"):
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass

    total_acc, total_prec, total_rec, total_f1, total_wer, total_ser = 0, 0, 0, 0, 0, 0
    num_samples = len(ground_truth_test_list)

    for pred, actual in zip(predicted_transcripts, ground_truth_test_list):
        acc, prec, rec, f1, wer_val, ser_val = calculate_metrics(pred, actual)
        total_acc += acc
        total_prec += prec
        total_rec += rec
        total_f1 += f1
        total_wer += wer_val
        total_ser += ser_val

    print(f"\n------------------------------------------------------------------", flush=True)
    print(f"FINAL TEST METRICS FOR {num_samples}-SAMPLE UNSEEN ATCOSIM SUBSET", flush=True)
    print(f"------------------------------------------------------------------", flush=True)
    print(f"Accuracy  : {total_acc / num_samples * 100:.2f}%", flush=True)
    print(f"Precision : {total_prec / num_samples * 100:.2f}%", flush=True)
    print(f"Recall    : {total_rec / num_samples * 100:.2f}%", flush=True)
    print(f"F1 Score  : {total_f1 / num_samples * 100:.2f}%", flush=True)
    print(f"WER       : {total_wer / num_samples * 100:.2f}%", flush=True)
    print(f"SER       : {total_ser / num_samples * 100:.2f}%", flush=True)
    print(f"===================================================================", flush=True)
    sys.stdout.flush()